In [1]:
%%time
# Задание 6. Проверка антиградиента бинарной кросс-энтропии

import warnings

warnings.filterwarnings("ignore")

import numpy as np
from scipy.special import expit

RANDOM_STATE = 21


def binary_log_loss(y, logits):
    """Считает бинарную кросс-энтропию для каждого объекта."""
    return np.logaddexp(0.0, logits) - y * logits


def negative_gradient(y, logits):
    return y - expit(logits)  # <-- вычисляет сигмоиду и превращает логиты в вероятности p.
    # Градиент бинарной кросс-энтропии по логиту равен p - y,
    # поэтому функция возвращает антиградиент y - p.


y_example = np.array([1.0, 0.0])
logits_example = np.full(2, np.log(0.2 / 0.8))
corrections = negative_gradient(y_example, logits_example)

epsilon = 1e-5
# формула проверяется численно
numerical_gradient = (
                             binary_log_loss(y_example, logits_example + epsilon)
                             - binary_log_loss(y_example, logits_example - epsilon)
                     ) / (2 * epsilon)

assert np.allclose(corrections, -numerical_gradient)  # сравнивает результат функции с отрицательным градиентом,
# приближённо вычисленным по изменению функции потерь
assert np.allclose(corrections,
                   [0.8, -0.2])  # проверяет ожидаемые значения для двух объектов с вероятностью 0,2 и метками 1 и 0

print("Вероятности:", expit(logits_example))
print("Истинные метки:", y_example.astype(int))
print("Антиградиент:", corrections)

Вероятности: [0.2 0.2]
Истинные метки: [1 0]
Антиградиент: [ 0.8 -0.2]
CPU times: total: 344 ms
Wall time: 367 ms


In [2]:
%%time
# Задание 6. Загрузка обучающей и валидационной выборок

from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
TARGET = "IsBadBuy"

df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_valid = pd.read_csv(DATA_DIR / "df_valid.csv")

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_valid = df_valid.drop(columns=[TARGET])
y_valid = df_valid[TARGET]

CPU times: total: 781 ms
Wall time: 834 ms


In [3]:
%%time
# Задание 6. Кодирование категорий и заполнение пропусков

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_columns = [
    "Auction", "Make", "Model", "Trim", "SubModel", "Color",
    "Transmission", "WheelType", "Nationality", "Size",
    "TopThreeAmericanName", "BYRNO", "VNZIP1", "VNST",
]

excluded_columns = {
    "RefId", "PurchDate", "WheelTypeID",
    "PRIMEUNIT", "AUCGUART", "IsOnlineSale",
}

numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns and column not in excluded_columns
]

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="__MISSING__"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = ColumnTransformer(
    [
        ("categorical", categorical_pipeline, categorical_columns),
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ],
    sparse_threshold=1.0,
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

assert X_train_encoded.shape[0] == len(y_train)
assert X_valid_encoded.shape[0] == len(y_valid)
assert X_train_encoded.shape[1] == X_valid_encoded.shape[1]

print(f"Train: {X_train_encoded.shape}, {type(X_train_encoded).__name__}")
print(f"Validation: {X_valid_encoded.shape}, {type(X_valid_encoded).__name__}")

Train: (24391, 1857), csr_matrix
Validation: (24255, 1857), csr_matrix
CPU times: total: 1.27 s
Wall time: 1.29 s


In [4]:
%%time

# Задание 6. Поиск порога для дерева регрессии

def best_regression_split(values, targets):
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_targets = np.asarray(targets, dtype=float)[order]

    cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:])
    if cuts.size == 0:
        return None, None

    left_size = cuts + 1
    right_size = len(values) - left_size

    # Центрирование помогает точнее вычислить дисперсию.
    centered = sorted_targets - sorted_targets.mean()
    prefix_sum = np.cumsum(centered)
    prefix_square_sum = np.cumsum(centered ** 2)

    left_sum = prefix_sum[cuts]
    right_sum = prefix_sum[-1] - left_sum
    left_square_sum = prefix_square_sum[cuts]
    right_square_sum = prefix_square_sum[-1] - left_square_sum

    left_variance = left_square_sum / left_size - (left_sum / left_size) ** 2
    right_variance = right_square_sum / right_size - (right_sum / right_size) ** 2

    left_std = np.sqrt(np.maximum(left_variance, 0.0))
    right_std = np.sqrt(np.maximum(right_variance, 0.0))
    scores = (left_size * left_std + right_size * right_std) / len(values)

    best = np.argmin(scores)
    cut = cuts[best]
    low = float(sorted_values[cut])
    high = float(sorted_values[cut + 1])
    threshold = low / 2 + high / 2

    return threshold, float(scores[best])

CPU times: total: 0 ns
Wall time: 0 ns


In [5]:
%%time
# Задание 6. Классы RegressionNode и DecisionTreeRegressor

from scipy.sparse import csc_matrix


class RegressionNode:
    def __init__(self, targets):
        self.prediction = float(np.mean(targets))
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None


class DecisionTreeRegressor:
    def __init__(self, max_depth=7, max_features=None, random_state=RANDOM_STATE):
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        X_csc = csc_matrix(X)
        if self.max_features is not None:
            if not isinstance(self.max_features, (int, np.integer)):
                raise TypeError("max_features должен быть целым числом или None")
            if not 1 <= self.max_features <= X_csc.shape[1]:
                raise ValueError("max_features должен быть от 1 до числа признаков")

        self._feature_rng = np.random.default_rng(self.random_state)
        targets = np.asarray(y, dtype=float)
        rows = np.arange(X_csc.shape[0])
        self.root_ = self._grow(X_csc, targets, rows, depth=0)
        return self

    def _best_node_split(self, X, targets, rows):
        node_targets = targets[rows]
        best_feature = None
        best_threshold = None
        best_score = np.inf

        features = range(X.shape[1])
        if self.max_features is not None:
            features = self._feature_rng.choice(
                X.shape[1], size=self.max_features, replace=False
            )

        for feature in features:
            values = X[rows, feature].toarray().ravel()
            threshold, score = best_regression_split(values, node_targets)

            if score is not None and score < best_score:
                best_feature = feature
                best_threshold = threshold
                best_score = score

        return best_feature, best_threshold, best_score

    def _grow(self, X, targets, rows, depth):
        node_targets = targets[rows]
        node = RegressionNode(node_targets)

        if depth >= self.max_depth or np.all(node_targets == node_targets[0]):
            return node

        feature, threshold, score = self._best_node_split(X, targets, rows)
        if feature is None or score >= node_targets.std():
            return node

        values = X[rows, feature].toarray().ravel()
        left_mask = values <= threshold

        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X, targets, rows[left_mask], depth + 1)
        node.right = self._grow(X, targets, rows[~left_mask], depth + 1)
        return node

    def predict(self, X):
        X_csc = csc_matrix(X)
        predictions = np.empty(X_csc.shape[0], dtype=float)

        def fill_predictions(node, rows):
            if rows.size == 0:
                return

            if node.feature is None:
                predictions[rows] = node.prediction
                return

            values = X_csc[rows, node.feature].toarray().ravel()
            left_mask = values <= node.threshold
            fill_predictions(node.left, rows[left_mask])
            fill_predictions(node.right, rows[~left_mask])

        fill_predictions(self.root_, np.arange(X_csc.shape[0]))
        return predictions

CPU times: total: 0 ns
Wall time: 0 ns


In [6]:
%%time
# Задание 6. Начальный логит и первый шаг градиентного бустинга

y_boost_train = np.asarray(y_train, dtype=float)
positive_rate = y_boost_train.mean()

initial_logit = np.log(positive_rate / (1.0 - positive_rate))
train_logits = np.full(y_boost_train.shape, initial_logit)
loss_before = binary_log_loss(y_boost_train, train_logits).mean()

residuals = negative_gradient(y_boost_train, train_logits)
first_tree = DecisionTreeRegressor(max_depth=1).fit(X_train_encoded, residuals)

learning_rate = 0.1
updated_logits = train_logits + learning_rate * first_tree.predict(X_train_encoded)
loss_after = binary_log_loss(y_boost_train, updated_logits).mean()

print(f"Начальная вероятность класса 1: {positive_rate:.6f}")
print(f"Начальный логит: {initial_logit:.6f}")
print(f"Кросс-энтропия до шага: {loss_before:.6f}")
print(f"Кросс-энтропия после шага: {loss_after:.6f}")

assert loss_after < loss_before

Начальная вероятность класса 1: 0.114673
Начальный логит: -2.043868
Кросс-энтропия до шага: 0.356176
Кросс-энтропия после шага: 0.354736
CPU times: total: 2.27 s
Wall time: 2.35 s


In [13]:
%%time

# Задание 6. Класс GBDTClassifier и последовательное обучение деревьев

class GBDTClassifier:
    def __init__(self, max_depth=2, number_of_trees=5, max_features=None,
                 learning_rate=0.1, random_state=RANDOM_STATE):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.random_state = random_state

    def fit(self, X, y):
        if not isinstance(self.number_of_trees, (int, np.integer)):
            raise TypeError("number_of_trees должен быть целым числом")
        if self.number_of_trees < 1:
            raise ValueError("number_of_trees должен быть не меньше 1")
        if not np.isfinite(self.learning_rate) or self.learning_rate <= 0:
            raise ValueError("learning_rate должен быть конечным положительным числом")

        X_csc = csc_matrix(X)
        targets = np.asarray(y, dtype=float)
        if X_csc.shape[1] == 0 or targets.shape != (X_csc.shape[0],):
            raise ValueError("X должен содержать признаки, а y должен содержать метку для каждой строки")
        if not np.array_equal(np.unique(targets), [0.0, 1.0]):
            raise ValueError("Для обучения нужны оба класса с метками 0 и 1")

        positive_rate = targets.mean()
        initial_logit = np.log(positive_rate / (1.0 - positive_rate))
        logits = np.full(targets.shape, initial_logit)
        train_losses = [binary_log_loss(targets, logits).mean()]
        trees = []

        tree_seeds = np.random.SeedSequence(self.random_state).spawn(self.number_of_trees)
        '''
        Инкрементальное обучение. На каждой итерации fit вычисляет антиградиент по текущим логитам, обучает новое регрессионное дерево, обновляет логиты и добавляет дерево в ансамбль. Следующая итерация использует обновлённые логиты.
        логит — числовой прогноз модели до перевода в вероятность
        
        1. negative_gradient(targets, logits) вычисляет поправки по текущему прогнозу ансамбля.
        2. DecisionTreeRegressor(...).fit(X_csc, residuals) создаёт и обучает новое дерево предсказывать эти поправки.
        3. logits += ... добавляет вклад нового дерева к текущему прогнозу. На следующей итерации поправки вычисляются по обновлённым логитам.
        4. trees.append(tree) добавляет обученное дерево в список. После цикла этот список сохраняется в self.trees_.
        
        '''
        for tree_seed in tree_seeds:
            residuals = negative_gradient(targets, logits)
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                max_features=self.max_features,
                random_state=tree_seed,
            ).fit(X_csc, residuals)
            logits += self.learning_rate * tree.predict(X_csc)
            trees.append(tree)
            train_losses.append(binary_log_loss(targets, logits).mean())

        self.initial_logit_ = initial_logit
        self.learning_rate_ = self.learning_rate
        self.n_features_in_ = X_csc.shape[1]
        self.trees_ = trees
        self.train_losses_ = np.asarray(train_losses)
        return self

    def decision_function(self, X):
        if not hasattr(self, "trees_"):
            raise ValueError("Сначала обучите ансамбль через fit")

        X_csc = csc_matrix(X)
        if X_csc.shape[1] != self.n_features_in_:
            raise ValueError("Число признаков должно совпадать с обучающей выборкой")

        logits = np.full(X_csc.shape[0], self.initial_logit_)
        for tree in self.trees_:
            logits += self.learning_rate_ * tree.predict(X_csc)
        return logits

    def predict_proba(self, X):
        positive = expit(self.decision_function(X))
        return np.column_stack((1.0 - positive, positive))

    def predict(self, X):
        return (self.decision_function(X) > 0.0).astype(int)

CPU times: total: 0 ns
Wall time: 0 ns


In [8]:
%%time
# Задание 6. Потери по итерациям и Gini на валидации

from sklearn.metrics import roc_auc_score

gbdt = GBDTClassifier(
    max_depth=2,
    number_of_trees=5,
    max_features=256,
    learning_rate=0.3,
    random_state=RANDOM_STATE,
).fit(X_train_encoded, y_train)

valid_proba = gbdt.predict_proba(X_valid_encoded)
assert len(gbdt.trees_) == gbdt.number_of_trees
assert valid_proba.shape == (len(y_valid), 2)
assert np.all((valid_proba >= 0) & (valid_proba <= 1))
assert np.allclose(valid_proba.sum(axis=1), 1.0)
assert gbdt.train_losses_[-1] < gbdt.train_losses_[0]

print(f"Бинарная кросс-энтропия на train до обучения: {gbdt.train_losses_[0]:.6f}")
for iteration, loss in enumerate(gbdt.train_losses_[1:], start=1):
    print(f"После дерева {iteration}: {loss:.6f}")

gbdt_valid_auc = roc_auc_score(y_valid, valid_proba[:, 1])
gbdt_valid_gini = 2 * gbdt_valid_auc - 1
print(f"ROC AUC на валидации: {gbdt_valid_auc:.4f}")
print(f"Gini на валидации: {gbdt_valid_gini:.4f}")

Бинарная кросс-энтропия на train до обучения: 0.356176
После дерева 1: 0.355422
После дерева 2: 0.354658
После дерева 3: 0.353961
После дерева 4: 0.353303
После дерева 5: 0.352601
ROC AUC на валидации: 0.6375
Gini на валидации: 0.2749
CPU times: total: 3.53 s
Wall time: 3.65 s
